# Test Query Team Execution

This notebook demonstrates two ways to run the query team workflow:
1. **Using QueryManager**: Simulates the standard way of submitting a query and getting the final result.
2. **Using Direct Graph Stream**: Directly interacts with the LangGraph instance to observe intermediate steps.

## Part 0: Setup (Imports and Ontology)

In [1]:
import sys
import os
sys.path.append(r"D:\CursorProj\Chem-Ontology-Constructor")
os.environ["PROJECT_ROOT"] = "D:\\CursorProj\\Chem-Ontology-Constructor\\"

from owlready2 import get_ontology
from config.settings import ONTOLOGY_CONFIG
# 加载本体文件
# onto1 = ONTOLOGY_CONFIG["ontology"]
# onto = get_ontology("data/ontology/test.owl").load() # 可选的第二个本体

In [2]:
# Required Imports
import sys
import os
import json
import time
from typing import Dict, Any, List
from owlready2 import *
import asyncio # Needed for owlready2 async operations in some envs

# Add project root to sys.path for imports
# Adjust the relative path if necessary based on where you run the notebook
# Assuming the notebook is in tests/unit_test/query/test/
# project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', '..'))
# if project_root not in sys.path:
#     sys.path.insert(0, project_root)
#     print(f"Added project root to path: {project_root}") # Verify path

# Check if the module can be found now
try:
    from autology_constructor.idea.query_team import QueryManager, Query, QueryStatus, create_query_graph
    from autology_constructor.idea.query_team.ontology_tools import OntologyTools
    from autology_constructor.idea.common.llm_provider import get_cached_default_llm
    print("Modules imported successfully.")
except ModuleNotFoundError as e:
    print(f"Error importing modules: {e}")
    print(f"Current sys.path: {sys.path}")
    # You might need to adjust the project_root calculation above or ensure __init__.py files exist

# Ensure LLM Provider is configured (e.g., set OPENAI_API_KEY environment variable)
try:
    llm = get_cached_default_llm()
    print("LLM Provider initialized successfully.")
except Exception as e:
    print(f"Error initializing LLM Provider: {e}\nPlease ensure API keys or necessary configurations are set.")
    llm = None

# --- Ontology Setup ---
# Option 1: Load an existing ontology file
# onto_path = "path/to/your/ontology.owl"
# test_onto = get_ontology(f"file://{os.path.abspath(onto_path)}").load()

# Option 2: Create a simple test ontology in memory
print("Creating a simple in-memory ontology...")
# It's good practice to clear existing ontologies from the default world if running cells repeatedly

for o in list(default_world.ontologies.values()): # <-- Corrected: iterate over values
    # Check if the ontology object has a callable __destroy__ method before attempting to destroy
    if callable(getattr(o, '__destroy__', None)):
        # print(f"Attempting to destroy ontology: {o.base_iri}") # Optional: for debugging
        try:
            destroy_entity(o)
        except Exception as destroy_err:
            print(f"Error destroying {o.base_iri}: {destroy_err}") # Log specific errors if needed
    else:
        print(f"Skipping destroy for non-callable __destroy__ or missing: {o.base_iri}") # Optional: for debugging
test_onto = get_ontology("data/ontology/test.owl").load()
for o in list(default_world.ontologies.values()):
    print(f"has：{o.base_iri}")


# print(f"Test Ontology '{test_onto.base_iri}' created with:\n- Classes: {[c.name for c in test_onto.classes()]}\n- Individuals: {[i.name for i in test_onto.individuals()]}\n- Object Properties: {[p.name for p in test_onto.object_properties()]}\n- Data Properties: {[p.name for p in test_onto.data_properties()]}")

# Run async tasks if needed by owlready2 backend
# try:
#     loop = asyncio.get_event_loop()
# except RuntimeError:
#     loop = asyncio.new_event_loop()
#     asyncio.set_event_loop(loop)
# loop.run_until_complete(asyncio.sleep(0)) # Run pending async tasks

Modules imported successfully.
LLM Provider initialized successfully.
Creating a simple in-memory ontology...
Skipping destroy for non-callable __destroy__ or missing: http://anonymous/
Skipping destroy for non-callable __destroy__ or missing: http://www.test.org/chem_ontologies/chem_ontology.owl#
has：http://anonymous/
has：http://www.test.org/chem_ontologies/chem_ontology.owl#
has：http://www.test.org/chem_ontologies/chem_ontology.owl#


## Part 1: Execution via QueryManager

### 中文问答对

**1. 问题 (难度：简单):** 本体中提到的一些笼状分子结构主要用在哪些方面？
   **答案：**
   *   结合阴离子 (Anion binding)
   *   形成离子通道 (Ion-channel formation)
   *   吸收气体 (Gas absorption)

**2. 问题 (难度：简单):** 哪些化学物质反应可以生成杯状的大环化合物？
   **答案：**
   *   苯酚 (Phenol)
   *   甲醛 (Formaldehyde)

**3. 问题 (难度：中等):** 本体描述了一种可以捕获和释放二羧酸客体的受体，它的组装和拆卸过程伴随着什么现象？
   **答案：**
   *   可见的颜色变化 (Visible color changes)

**4. 问题 (难度：中等):** 本体中提到了哪些分子间相互作用力与一种特定的笼状分子和溶剂（如二氯甲烷和水）的结合有关？
   **答案：**
   *   氢键 (Hydrogen bonding)
   *   卤键 (Halogen bonding)
   *   CH···π 相互作用 (CH···π interaction)

**5. 问题 (难度：中等):** 哪种化学合成策略被用来制备多孔有机笼？
   **答案：**
   *   动态共价化学 (Dynamic covalent chemistry, DCC)

**6. 问题 (难度：中等):** 在合成某种双杯[4]吡咯衍生物时，使用了什么物质来提高产率或效率？
   **答案：**
   *   对苯二甲酸四丁基铵 (Tetrabutylammonium terephthalate) 被用作模板 (template)。

**7. 问题 (难度：困难):** 本体中描述了哪种分子能够帮助氨基酸（如脯氨酸）穿过细胞膜？
   **答案：**
   *   一种称为单膦酸酯杯[4]芳烃的分子 (Monophosphonate cavitand 8a)。

**8. 问题 (难度：困难):** 本体中描述的一种复杂的寡吡咯受体是如何与离子或其他小分子结合的？描述其结构特点。
   **答案：**
   *   它充当特定阴离子的主体 (supramolecular host)。
   *   结构上包含提供氢键的基团（酰胺和吡咯质子）(hydrogen-bond donors: amidic and pyrrolic protons)。
   *   结构上包含接受氢键的基团（亚胺氮原子）(hydrogen-bond acceptors: imine nitrogen atoms)。

**9. 问题 (难度：困难):** 阴离子与一些相互锁定的分子结构（如索烃或轮烷）结合时，会产生哪些影响？
   **答案：**
   *   阴离子（如卤化物）可在合成中起模板作用 (template)。
   *   可以实现对特定阴离子（如氯离子）的选择性识别 (selective recognition)。
   *   可能引起分子结构的运动或构象变化（如环旋转或梭动）(ring rotation or shuttling motion)。
   *   可以改变分子的电化学或光学性质（如发光增强或波长改变）(affect electrochemical/optical properties)。

**10. 问题 (难度：非常困难):** 描述一下本体中提到的某种“四壁”杯[4]吡咯衍生物在模拟细胞膜环境中转运阴离子的机制。
    **答案：**
    *   它充当分子载体 (molecular carrier)。
    *   转运研究常在脂质体囊泡 (liposomal vesicles) 中进行。
    *   其机制通常涉及阴离子/阴离子反向转运 (anion/anion antiport mechanism)，即一个阴离子进入囊泡伴随另一个阴离子移出。

### English Question-Answer Pairs

**1. Question (Difficulty: Simple):** What are some main application areas mentioned in the ontology for certain cage-like molecular structures?
   **Answer:**
   *   Binding anions (Anion binding)
   *   Forming ion channels (Ion-channel formation)
   *   Absorbing gases (Gas absorption)

**2. Question (Difficulty: Simple):** Which chemical substances react to form cup-shaped macrocyclic compounds?
   **Answer:**
   *   Phenol
   *   Formaldehyde

**3. Question (Difficulty: Medium):** The ontology describes a receptor that can capture and release dicarboxylic acid guests. What phenomenon accompanies its assembly and disassembly process?
   **Answer:**
   *   Visible color changes

**4. Question (Difficulty: Medium):** What types of intermolecular forces are mentioned in the ontology regarding the interaction between a specific cage molecule and solvents like dichloromethane and water?
   **Answer:**
   *   Hydrogen bonding
   *   Halogen bonding
   *   CH···π interaction

**5. Question (Difficulty: Medium):** Which chemical synthesis strategy is used to prepare porous organic cages?
   **Answer:**
   *   Dynamic covalent chemistry (DCC)

**6. Question (Difficulty: Medium):** What substance was used to improve the yield or efficiency during the synthesis of a certain bis-calix[4]pyrrole derivative?
   **Answer:**
   *   Tetrabutylammonium terephthalate was used as a template.

**7. Question (Difficulty: Difficult):** Which molecule is described in the ontology as capable of helping amino acids (like proline) pass through cell membranes?
   **Answer:**
   *   A molecule referred to as a monophosphonate cavitand (specifically 8a).

**8. Question (Difficulty: Difficult):** How does a complex oligopyrrolic receptor described in the ontology bind with ions or other small molecules? Describe its structural features related to binding.
   **Answer:**
   *   It acts as a supramolecular host for specific anions.
   *   Structurally, it contains groups that donate hydrogen bonds (amidic and pyrrolic protons).
   *   Structurally, it contains groups that accept hydrogen bonds (imine nitrogen atoms).

**9. Question (Difficulty: Difficult):** What effects arise when anions bind to some interlocked molecular structures (like catenanes or rotaxanes) mentioned in the ontology?
   **Answer:**
   *   Anions (e.g., halides) can act as templates during synthesis.
   *   Selective recognition of specific anions (e.g., chloride) can be achieved.
   *   It may induce movement or conformational changes in the molecular structure (e.g., ring rotation or shuttling motion).
   *   It can alter the electrochemical or optical properties of the molecule (e.g., enhance luminescence or shift wavelength).

**10. Question (Difficulty: Very Difficult):** Describe the mechanism by which a certain "four-wall" calix[4]pyrrole derivative, mentioned in the ontology, transports anions in a simulated cell membrane environment.
    **Answer:**
    *   It functions as a molecular carrier.
    *   Transport studies are often conducted in liposomal vesicles.
    *   The mechanism typically involves an anion/anion antiport process, where the entry of one anion into the vesicle is coupled with the exit of another.

In [3]:
if not llm:
    print("Skipping QueryManager test due to LLM initialization failure.")
else:
    print("--- Starting QueryManager Test ---")
    query_manager = QueryManager()

    # 更新类缓存
    print("Updating class name cache...")
    query_manager.update_class_name_cache(test_onto)
    # 打印部分缓存内容以确认
    if query_manager.class_name_cache:
         print(f"Cache content (first 10): {query_manager.class_name_cache[:10]}...")
    else:
         print("Class name cache is empty.")


    # 启动管理器
    print("Starting QueryManager...")
    query_manager.start()

    # 定义新的十个查询
    queries = [
        # Simple
        "What are some main application areas mentioned in the ontology for certain cage-like molecular structures?",
        # "Which chemical substances react to form cup-shaped macrocyclic compounds?",
        # # Medium
        # "The ontology describes a receptor that can capture and release dicarboxylic acid guests. What phenomenon accompanies its assembly and disassembly process?",
        # "What types of intermolecular forces are mentioned in the ontology regarding the interaction between a specific cage molecule and solvents like dichloromethane and water?",
        # "Which chemical synthesis strategy is used to prepare porous organic cages?",
        # "What substance was used to improve the yield or efficiency during the synthesis of a certain bis-calix[4]pyrrole derivative?",
        # # Difficult
        # "Which molecule is described in the ontology as capable of helping amino acids (like proline) pass through cell membranes?",
        # "How does a complex oligopyrrolic receptor described in the ontology bind with ions or other small molecules? Describe its structural features related to binding.",
        # "What effects arise when anions bind to some interlocked molecular structures (like catenanes or rotaxanes) mentioned in the ontology?",
        # # Very Difficult
        # "Describe the mechanism by which a certain \"four-wall\" calix[4]pyrrole derivative, mentioned in the ontology, transports anions in a simulated cell membrane environment."
    ]

    # 为所有查询定义统一的上下文
    query_context = {
        "ontology": test_onto,
        "originating_team": "test_notebook",
        "originating_stage": "manual_test",
        "query_type": "information_retrieval" # 对所有查询使用信息检索类型
    }

    # 提交多个查询并收集futures
    futures = []
    print(f"\\nSubmitting {len(queries)} queries...")
    for i, query_text in enumerate(queries):
        print(f"Submitting query {i+1}: '{query_text[:80]}...'") # 打印部分查询文本
        future = query_manager.submit_query(query_text=query_text, query_context=query_context)
        futures.append((i+1, query_text, future))

    print("\nAll queries submitted.")

--- Starting QueryManager Test ---
Updating class name cache...
Class name cache updated with 1065 classes.
Cache content (first 10): ['(Co10L15)20+ pentagonal prism', '(Co4L6)8+ tetrahedron', '(Fe4L6)8+ tetrahedral cage', '1,2,3-triazolium_compound', '1,3,5-triethyl-2,4,6-trimethylamine', '1,3,5-trisubstituted-2,4,6-triethylbenzene scaffold', '1,3-alternate_tetrabutylamide_calix(4)arene', '1,3-diynyl_linker', '1,4-triazole_linker', '1-palmitoyl-2-oleoyl-sn-glycero-3-phosphocholine(POPC)']...
Starting QueryManager...
0
0-1
Dispatcher loop started on thread QueryDispatcherThread
Query Manager dispatcher started.
\nSubmitting 1 queries...
Submitting query 1: 'What are some main application areas mentioned in the ontology for certain cage-...'

All queries submitted.
1
2
3


4:  intent='find information' target_entities=['cage-like molecular structures'] properties=['application areas'] filters=None query_type_suggestion='fact-finding'
7
Query processing error: Cannot instantiate typing.Union


In [4]:
if not llm:
    print("Skipping QueryManager test due to LLM initialization failure.")
else:    
    # 等待并获取所有结果
    print("Waiting for queries completion...")
    try:
        for i, query_text, future in futures:
            print(f"\nProcessing results for query {i}: '{query_text}'")
            # 等待合理的时间（根据需要调整）
            final_result_dict = future.result(timeout=120)
            print(f"Query {i} completed.")
            # 美观打印最终状态字典
            print(f"\n--- Final State Dictionary for Query {i} ---")
            # 使用default=str处理潜在的不可序列化对象，如本体引用
            print(json.dumps(final_result_dict, indent=2, default=str))
    except Exception as e:
        print(f"Error getting query result: {e}")
        if future.done() and future.exception():
             print(f"Future exception details: {future.exception()}")
    finally:
        # 停止管理器
        print("\nStopping QueryManager...")
        query_manager.stop()
        print("QueryManager stopped.")

    print("--- QueryManager Test Finished ---")


Waiting for queries completion...

Processing results for query 1: 'What are some main application areas mentioned in the ontology for certain cage-like molecular structures?'
Error getting query result: Cannot instantiate typing.Union
Future exception details: Cannot instantiate typing.Union

Stopping QueryManager...
Stopping Query Manager...
Shutting down query executor...
Query Manager stopped.
QueryManager stopped.
--- QueryManager Test Finished ---


**Note on Streaming with QueryManager:**

The standard `QueryManager.submit_query()` returns a `Future` that resolves to the *final* state of the LangGraph execution. It doesn't inherently provide access to the intermediate states generated by each node.

To observe the step-by-step execution and intermediate state changes, you would typically need to interact directly with the LangGraph instance using its `stream()` method, as demonstrated in Part 2 below. Modifying the `QueryManager` to expose this stream would require significant changes to its asynchronous task handling and result reporting.

## Part 2: Direct Execution via Graph Stream

In [ ]:
if not llm:
    print("Skipping Direct Graph Stream test due to LLM initialization failure.")
else:
    print("\n--- Starting Direct Graph Stream Test ---")

    # 1. Create graph instance
    print("Creating graph instance...")
    graph = create_query_graph()
    print("Graph instance created.")

    # 2. Manually create initial state dictionary
    print("Creating initial state...")
    # Use the same query as Part 1 for comparison
    # query_text_stream = "What proteins does DrugA bind to?"
    query_text_stream = "What are some main application areas mentioned in the ontology for certain cage-like molecular structures?"

    try:
        # Ensure we get a list of strings
        available_classes_stream = sorted([cls.name for cls in test_onto.classes() if isinstance(cls, ThingClass)])
    except Exception as e:
        print(f"Error getting class names: {e}")
        available_classes_stream = []

    initial_state = {
        "query": query_text_stream,
        "source_ontology": test_onto, # Pass the actual ontology object
        "available_classes": available_classes_stream,
        # Add other necessary fields, initializing appropriately
        "query_type": "information_retrieval",
        "query_strategy": None,
        "additional_ontology": None,
        "originating_team": "test_notebook_stream",
        "originating_stage": "manual_stream_test",
        "query_results": {},
        "normalized_query": None,
        "execution_plan": None,
        "validation_report": None,
        "sparql_query": None,
        "status": "initialized",
        "stage": "initialized",
        "previous_stage": None,
        "error": None,
        "messages": [] # LangGraph expects messages field
    }
    print("Initial state prepared.")
    # print(json.dumps(initial_state, indent=2, default=str)) # Optionally print initial state (ontology won't serialize well)

    # 3. Execute and iterate stream
    print("\n--- Streaming Graph Execution --- ")
    try:
        stream_counter = 0
        # Use stream method to get intermediate steps
        for chunk in graph.stream(initial_state):
            stream_counter += 1
            print(f"\n--- Chunk {stream_counter} --- ")
            # Chunks are dictionaries where keys are node names that just ran
            # and values are the outputs (state updates) returned by that node
            # Use default=str to handle potential non-serializable objects in the state
            print(json.dumps(chunk, indent=2, default=str))
            print("-" * 30)
        print("\n--- Graph Stream Finished --- ")
    except Exception as e:
        print(f"\nError during graph stream: {e}")
        import traceback
        traceback.print_exc() # Print full traceback for stream errors

    print("--- Direct Graph Stream Test Finished ---")


--- Starting Direct Graph Stream Test ---
Creating graph instance...
Graph instance created.
Creating initial state...
Initial state prepared.

--- Streaming Graph Execution --- 

--- Chunk 1 --- 
{
  "normalize": {
    "normalized_query": "intent='find information' target_entities=['cage-like molecular structures'] properties=['application areas'] filters=None query_type_suggestion='fact-finding'",
    "status": "parsing_complete",
    "stage": "normalized",
    "previous_stage": "initialized",
    "messages": [
      "content='Query normalized: What are some main application areas mentioned in the ontology for certain cage-like molecular structures?' additional_kwargs={} response_metadata={} id='dae860e2-906e-4763-8841-0e2f4af11652'"
    ]
  }
}
------------------------------

--- Chunk 2 --- 
{
  "strategy": {
    "query_strategy": "tool_sequence",
    "status": "strategy_determined",
    "stage": "strategy",
    "previous_stage": "normalized",
    "messages": [
      "content='Que

In [4]:
available_classes_stream

['(Co10L15)20+ pentagonal prism',
 '(Co4L6)8+ tetrahedron',
 '(Fe4L6)8+ tetrahedral cage',
 '1,2,3-triazolium_compound',
 '1,3,5-triethyl-2,4,6-trimethylamine',
 '1,3,5-trisubstituted-2,4,6-triethylbenzene scaffold',
 '1,3-alternate_tetrabutylamide_calix(4)arene',
 '1,3-diynyl_linker',
 '1,4-triazole_linker',
 '1-palmitoyl-2-oleoyl-sn-glycero-3-phosphocholine(POPC)',
 '10-camphorsulfonate',
 '1:1_inclusion_complex',
 '1H_NMR_spectroscopic_titration',
 '2,6-bis(arylethynyl)pyridine',
 '2-formylpyridine',
 '2catenane',
 '2rotaxane',
 '31_LCG_complex',
 '4,4’-diamide-2,2’-bipyridyl macrocycle',
 '4,4’-diaminobiphenyl',
 '4-methylbenzylhydrazide',
 '5,6-dichloro-2,3-dicyano-p-benzoquinone(DDQ)',
 '5-carboxyfluorescein',
 '6,6’-diformyl-3,3’-bipyridine',
 '8-hydroxy-1,3,6-pyrenetrisulfonate(HPTS)_assay',
 '8a',
 'ACh-responsive_system',
 'AE-C(4)P',
 'AE-C(4)Ps',
 'Au@Pd_nanoparticle',
 'BT-549_cell',
 'C(4)P_unit',
 'C4A',
 'C4A-based_nanoaggregate',
 'C4A_derivative',
 'C8A',
 'CA(calixar